In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [12]:
df=pd.read_csv('preprocessed_data.csv')

In [13]:
df.head()

,Gender,Academic Pressure,CGPA,Study Satisfaction,Sleep Duration,Dietary Habits,suicidal_thoughts,Work/Study Hours,Depression,Degree_B.Arch,...,Degree_M.Tech,Degree_MA,Degree_MBA,Degree_MBBS,Degree_MCA,Degree_MD,Degree_ME,Degree_MHM,Degree_MSc,Degree_PhD
0,1,5,8.97,2,1,2,1,3,1,False,...,False,False,False,False,False,False,False,False,False,False
1,2,2,5.90,5,1,1,0,3,0,False,...,False,False,False,False,False,False,False,False,False,False
2,1,3,7.03,5,0,2,0,9,0,False,...,False,False,False,False,False,False,False,False,False,False
3,2,3,5.59,2,2,1,1,4,1,False,...,False,False,False,False,False,False,False,False,False,False
4,2,4,8.13,3,1,1,1,1,0,False,...,True,False,False,False,False,False,False,False,False,False


In [14]:
X=df.drop(columns='Depression')
y=df['Depression']

In [15]:
X.head()

,Gender,Academic Pressure,CGPA,Study Satisfaction,Sleep Duration,Dietary Habits,suicidal_thoughts,Work/Study Hours,Degree_B.Arch,Degree_B.Com,...,Degree_M.Tech,Degree_MA,Degree_MBA,Degree_MBBS,Degree_MCA,Degree_MD,Degree_ME,Degree_MHM,Degree_MSc,Degree_PhD
0,1,5,8.97,2,1,2,1,3,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2,2,5.90,5,1,1,0,3,False,False,...,False,False,False,False,False,False,False,False,False,False
2,1,3,7.03,5,0,2,0,9,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2,3,5.59,2,2,1,1,4,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2,4,8.13,3,1,1,1,1,False,False,...,True,False,False,False,False,False,False,False,False,False


In [16]:
y.head()

0    1
1    0
2    0
3    1
4    0
Name: Depression, dtype: int64

In [17]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [18]:
X_train.shape

(19455, 35)

In [19]:
y_train.shape

(19455,)

In [21]:
X_test.shape

(8338, 35)

Since our data is not normally distributed we are using MinMaxScaler, which will scale the value between 0 and 1

In [22]:
MinMaxScaler=MinMaxScaler()
X_train=MinMaxScaler.fit_transform(X_train)
X_test=MinMaxScaler.transform(X_test)

In [24]:
X_train

array([[0.   , 0.   , 0.924, ..., 0.   , 0.   , 0.   ],
       [1.   , 0.25 , 0.708, ..., 0.   , 0.   , 0.   ],
       [1.   , 0.25 , 0.56 , ..., 0.   , 0.   , 1.   ],
       ...,
       [0.   , 0.   , 0.728, ..., 0.   , 0.   , 0.   ],
       [1.   , 0.25 , 0.987, ..., 0.   , 0.   , 0.   ],
       [0.   , 1.   , 0.551, ..., 0.   , 0.   , 0.   ]])

Now we will implement Machine Learning and Deep learning models and calculate the train and test accuracies, so that we can see if there is overfitting or underfitting. We will keep 5% difference as our threshold for overfitting scenario. If that happens we will use regularization techniques to generalize our model.

##### Machine Learning Models

In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
model=LogisticRegression()
model.fit(X_train,y_train)

y_pred=model.predict(X_test)
y_pred_train=model.predict(X_train)
test_accuracy=np.round(accuracy_score(y_test,y_pred)*100,2)
train_accuracy=np.round(accuracy_score(y_train,y_pred_train)*100,2)
print('Test Accuracy:',test_accuracy)
print('Train Accuracy:',train_accuracy)
print('Difference:',round(abs(test_accuracy-train_accuracy),2))

Test Accuracy: 81.87
Train Accuracy: 82.66
Difference: 0.79


In [43]:
from sklearn.tree import DecisionTreeClassifier

DTC_model=DecisionTreeClassifier(random_state=42)
DTC_model.fit(X_train,y_train)

y_pred_DTC_test=DTC_model.predict(X_test)
y_pred_DTC_train=DTC_model.predict(X_train)

test_accuracy_DTC=np.round(accuracy_score(y_test,y_pred_DTC_test)*100,2)
train_accuracy_DTC=np.round(accuracy_score(y_train,y_pred_DTC_train)*100,2)

print('Test Accuracy:',test_accuracy_DTC)
print('Train Accuracy:',train_accuracy_DTC)
print('Difference:',round(abs(test_accuracy_DTC-train_accuracy_DTC),2))


Test Accuracy: 74.47
Train Accuracy: 99.99
Difference: 25.52


Our Decision Tree heavily overfits. So we will try fine-tuning the parameters and then run the model.

We are first doing randomized search to get a set of best parameters, then using that parameters in gridsearch to get even a better set of parameters, doing this will increase the accuracy of the decision tree and it will be computationally less expensive. 

In [36]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV


In [42]:
random_DTC_params={
    'max_depth': [None, 3, 5, 10, 20],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'criterion': ['gini', 'entropy'],
    'max_features': [None, 'sqrt', 'log2']
}

DTC_rs=RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_distributions=random_DTC_params,
    n_iter=10,
    cv=3,
    random_state=42,
    n_jobs=-1
)

DTC_rs.fit(X_train,y_train)
print('Best parameters using Randomized Search:',DTC_rs.best_params_)

DTC_rs_params=DTC_rs.best_params_


Best parameters using Randomized Search: {'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': None, 'max_depth': 10, 'criterion': 'entropy'}


In [48]:
grid_DTC_params={
    'max_depth': [DTC_rs_params['max_depth']],
    'min_samples_split': [DTC_rs_params['min_samples_split']-1,DTC_rs_params['min_samples_split'],DTC_rs_params['min_samples_split']+1],
    'min_samples_leaf': [DTC_rs_params['min_samples_leaf']-1,DTC_rs_params['min_samples_leaf'],DTC_rs_params['min_samples_leaf']+1],
    'criterion': [DTC_rs_params['criterion']],
    'max_features': [DTC_rs_params['max_features']]
}

DTC_gs=GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=grid_DTC_params,
    cv=3,
    n_jobs=-1    
)

DTC_gs.fit(X_train,y_train)
print('Best parameters using Grid Search:',DTC_gs.best_params_)

best_DTC_gs=DTC_gs.best_estimator_

y_pred_DTC_test_gs=best_DTC_gs.predict(X_test)
y_pred_DTC_train_gs=best_DTC_gs.predict(X_train)

test_accuracy_DTC_gs=np.round(accuracy_score(y_test,y_pred_DTC_test_gs)*100,2)
train_accuracy_DTC_gs=np.round(accuracy_score(y_train,y_pred_DTC_train_gs)*100,2)

print('Test Accuracy:',test_accuracy_DTC_gs)
print('Train Accuracy:',train_accuracy_DTC_gs)
print('Difference:',round(abs(test_accuracy_DTC_gs-train_accuracy_DTC_gs),2))

Best parameters using Grid Search: {'criterion': 'entropy', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 15}
Test Accuracy: 80.56
Train Accuracy: 84.41
Difference: 3.85


We can see that the test accuracy is increased a lot after fine-tuning the parameters. But still, the accuracy is not great compared to a normal logistic Regression. Now we will try fine-tuning the parameters for logistic Regression and then see if we get better accuracy

In [50]:
#Logistic Regression fine-tuning parameters. We will follow the same strategy for paramter finetuning
random_LR_params={
    'penalty':['l1','l2','none','elasticnet'],
    'solver': ['liblinear','lbfgs','saga',],
    'max_iter':[500,1000,100],
    'l1_ratio': [0.2,0.5,0.7]
}

LR_rs=RandomizedSearchCV(
    LogisticRegression(random_state=42),
    param_distributions=random_LR_params,
    n_iter=10,
    cv=3,
    random_state=42,
    n_jobs=-1
)

LR_rs.fit(X_train,y_train)
print('Best parameters for LR using RS:',LR_rs.best_params_)


c:\Users\athar\Projects\Statistics Project\Student_Depression_Analysis\venv\lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
12 fits failed out of a total of 30.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\athar\Projects\Statistics Project\Student_Depression_Analysis\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\athar\Projects\Statistics Project\Student_Depression_Analysis\venv\lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\

Best parameters for LR using RS: {'solver': 'saga', 'penalty': 'l1', 'max_iter': 100, 'l1_ratio': 0.5}


We got few errors above and that's not an issue because some parameters are compatible with some other parameters. Such as l1_ratio is compatible only when elasticnet is used. Otherwise it is not used. Similarly I am attaching the table of solvers compatibility with regularizations.<br>
| Penalty     | Compatible Solvers         |
|-------------|----------------------------|
| 'l2'      | 'lbfgs', 'newton-cg', 'sag', 'saga' |
| 'l1'      | 'liblinear', 'saga'     |
| 'elasticnet' | 'saga' only              |
| 'none'    | 'lbfgs', 'newton-cg', 'sag', 'saga' |

In [56]:
grid_LR_params={
    'penalty':[LR_rs.best_params_['penalty']],
    'solver': [LR_rs.best_params_['solver'],'liblinear'],
    'max_iter':[LR_rs.best_params_['max_iter'],LR_rs.best_params_['max_iter']+100,LR_rs.best_params_['max_iter']+200],
    'l1_ratio': [LR_rs.best_params_['l1_ratio'],LR_rs.best_params_['l1_ratio']-0.1,LR_rs.best_params_['l1_ratio']+0.1]
}

LR_gs=GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid=grid_LR_params,
    cv=3,
    n_jobs=-1
)

LR_gs.fit(X_train,y_train)
print('Best parameters for LR using grid search:', LR_gs.best_params_)

best_LR_gs=LR_gs.best_estimator_

y_pred_LR_test_gs=best_LR_gs.predict(X_test)
y_pred_LR_train_gs=best_LR_gs.predict(X_train)

test_accuracy_LR_gs=np.round(accuracy_score(y_test,y_pred_LR_test_gs)*100,2)
train_accuracy_LR_gs=np.round(accuracy_score(y_train,y_pred_LR_train_gs)*100,2)

print('Test Accuracy:',test_accuracy_LR_gs)
print('Train Accuracy:',train_accuracy_LR_gs)
print('Difference:',round(abs(test_accuracy_LR_gs-train_accuracy_LR_gs),2))


c:\Users\athar\Projects\Statistics Project\Student_Depression_Analysis\venv\lib\site-packages\sklearn\linear_model\_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Best parameters for LR using grid search: {'l1_ratio': 0.5, 'max_iter': 100, 'penalty': 'l1', 'solver': 'saga'}
Test Accuracy: 81.87
Train Accuracy: 82.68
Difference: 0.81


Test Accuracy after grid search was no different than the accuracy of logistic Regression with default parameters. We will try using RandomForest as our bagging technique and then we can use boosting techniques if needed. Additionally, if we feel that the accuracy is not the good performance metric we can see which performance metric is better. We will do a hypothesis testing to decide which error is more important and then decide performance metric.

In [57]:
from sklearn.ensemble import RandomForestClassifier

In [60]:
RFC_model= RandomForestClassifier(random_state=42)

RFC_model.fit(X_train,y_train)
y_pred_RFC_test=RFC_model.predict(X_test)
y_pred_RFC_train=RFC_model.predict(X_train)

RFC_test_accuracy=np.round(accuracy_score(y_test,y_pred_RFC_test)*100,2)
RFC_train_accuracy=np.round(accuracy_score(y_train,y_pred_RFC_train)*100,2)

print('Test accuracy:',RFC_test_accuracy)
print('Train accuracy:',RFC_train_accuracy)
print('Difference between test accuracy and train accuracy:',np.round(abs(RFC_test_accuracy-RFC_train_accuracy),2))

Test accuracy: 81.09
Train accuracy: 99.99
Difference between test accuracy and train accuracy: 18.9


Our RandomForest Classifier is overfitting with 19% difference. So we will do a parameter fine tuning and we will try to reduce the overfitting. We can toggle the parameters as per below:<br>
| Parameter              | Description                                                                 | Regularization Effect                           | Effect (More vs. Less) on Overfitting                  |
|------------------------|-----------------------------------------------------------------------------|--------------------------------------------------|--------------------------------------------------------|
| n_estimators         | Number of trees in the forest                                               | More trees reduce variance (helps generalize)   | **More** → ↓ overfitting (up to a point); **Less** → ↑ over fitting |
| max_depth            | Maximum depth of each tree                                                  | Limits overfitting by controlling complexity    | **More** → ↑ overfitting; **Less** → ↓ overfitting     |
| min_samples_split    | Minimum samples required to split a node                                    | Prevents overly complex trees                   | **More** → ↓ overfitting; **Less** → ↑ overfitting     |
| min_samples_leaf     | Minimum samples required at a leaf node                                     | Smoothens the model, reduces variance           | **More** → ↓ overfitting; **Less** → ↑ overfitting     |
| max_features         | Number of features to consider when looking for the best split              | Reduces correlation among trees (diversity)     | **More** → ↑ overfitting; **Less** → ↓ overfitting (adds randomness) |
| bootstrap            | Whether bootstrap samples are used when building trees                      | Adds randomness, helps generalization           | **True** → ↓ overfitting; **False** → ↑ risk of overfitting |

In [61]:
RS_RFC_params={
    'n_estimators':[100,200,300,400],
    'max_depth': [5,10,15],
    'min_samples_split': [7,8,9],
    'min_samples_leaf': [3,4,5],
    'max_features': ['sqrt','log2']
}

RFC_rs= RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=RS_RFC_params,
    n_iter=10,
    cv=3,
    random_state=42,
    n_jobs=-1
)

RFC_rs.fit(X_train,y_train)
print('Best parameters:', RFC_rs.best_params_)

Best parameters: {'n_estimators': 400, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 10}


In [62]:
gs_RFC_params={
    'n_estimators':[RFC_rs.best_params_['n_estimators'],RFC_rs.best_params_['n_estimators']+100,RFC_rs.best_params_['n_estimators']+200],
    'max_depth': [RFC_rs.best_params_['max_depth']-2,RFC_rs.best_params_['max_depth'],RFC_rs.best_params_['max_depth']+2],
    'min_samples_split': [RFC_rs.best_params_['min_samples_split']-2,RFC_rs.best_params_['min_samples_split'],RFC_rs.best_params_['min_samples_split']+2],
    'min_samples_leaf': [RFC_rs.best_params_['min_samples_leaf'],RFC_rs.best_params_['min_samples_leaf']+1,RFC_rs.best_params_['min_samples_leaf']+2],
    'max_features': [RFC_rs.best_params_['max_features']]    
}

RFC_gs=GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid= gs_RFC_params,
    cv=3,
    n_jobs=-1
)

RFC_gs.fit(X_train,y_train)
best_RFC_gs=RFC_gs.best_estimator_

y_pred_RFC_test_gs=best_RFC_gs.predict(X_test)
y_pred_RFC_train_gs=best_RFC_gs.predict(X_train)

test_accuracy_RFC_gs=np.round(accuracy_score(y_test,y_pred_RFC_test_gs)*100,2)
train_accuracy_RFC_gs=np.round(accuracy_score(y_train,y_pred_RFC_train_gs)*100,2)

print('Test Accuracy:',test_accuracy_RFC_gs)
print('Train Accuracy:',train_accuracy_RFC_gs)
print('Difference:',round(abs(test_accuracy_RFC_gs-train_accuracy_RFC_gs),2))

Test Accuracy: 81.75
Train Accuracy: 84.13
Difference: 2.38


We will try support vector classifier and mellow down the regularization.

In [78]:
from sklearn.svm import SVC
svc_model=SVC(random_state=42,C=15, kernel='rbf', gamma='auto')
svc_model.fit(X_train,y_train)
y_pred_svc_test= svc_model.predict(X_test)
y_pred_svc_train= svc_model.predict(X_train)

svc_test_accuracy=np.round(accuracy_score(y_test,y_pred_svc_test)*100,2)
svc_train_accuracy=np.round(accuracy_score(y_train,y_pred_svc_train)*100,2)

print('Test accuracy using SVC:',svc_test_accuracy)
print('Train accuracy using SVC:', svc_train_accuracy)
print('Difference:',np.round(abs(svc_test_accuracy-svc_train_accuracy),2))

Test accuracy using SVC: 81.78
Train accuracy using SVC: 82.66
Difference: 0.88


We will use the boosting method now

| Algorithm   | Best For                        | Speed   | Handles Categorical | Task Type           | Notes                              |
|-------------|----------------------------------|---------|----------------------|----------------------|-------------------------------------|
| **AdaBoost**    | Clean, small datasets           | Yes       | No                   | Classification only  | Simple, but noise-sensitive         |
| **GBM**         | Flexible, moderate datasets     | No       | No                   | Regression & Classification | Custom loss, slower                 |
| **XGBoost**     | Large tabular datasets          | Better    | No (use encoding)    | Regression & Classification | High performance, many features     |
| **LightGBM**    | Big data, many features         | Best |  (can be biased)   | Regression & Classification | Fastest, needs careful tuning for categories |
| **CatBoost**    | Categorical-heavy data          | Better     | Best               | Regression & Classification | Easiest for mixed-type datasets     |

From above chart we can say that XGBoost can be our best fit

In [82]:
from xgboost import XGBClassifier

In [98]:
XGB_model=XGBClassifier(random_state=42,eval_metric='logloss', alpha=5, n_estimators=20)

XGB_model.fit(X_train,y_train)

y_pred_XGB_test=XGB_model.predict(X_test)
y_pred_XGB_train=XGB_model.predict(X_train)

XGB_test_accuracy=np.round(accuracy_score(y_test,y_pred_XGB_test)*100,2)
XGB_train_accuracy=np.round(accuracy_score(y_train,y_pred_XGB_train)*100,2)

print('Test accuracy for XGB:',XGB_test_accuracy)
print('Train accuracy forXGB:',XGB_train_accuracy)
print('Difference:', np.round(abs(XGB_test_accuracy-XGB_train_accuracy),2))

Test accuracy for XGB: 81.65
Train accuracy forXGB: 83.73
Difference: 2.08


| **Performance Metric**      | **Focus on Type-1 Error (False Positive)** | **Focus on Type-2 Error (False Negative)** | **Formula**                                                                                      | **When to Prioritize Type-1 or Type-2 Error**                                              |
|-----------------------------|--------------------------------------------|--------------------------------------------|--------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------|
| **Accuracy**                | Moderate                                  | Moderate                                  | \( \frac{TP + TN}{TP + TN + FP + FN} \)                                                          | When the dataset is balanced (equal number of positives and negatives).                   |
| **Precision**               | Yes (minimizes False Positives)            | No                                         | \( \frac{TP}{TP + FP} \)                                                                          | When False Positives (misclassifying healthy students as depressed) are more costly or harmful. |
| **Recall (Sensitivity)**    | No                                         | Yes (minimizes False Negatives)            | \( \frac{TP}{TP + FN} \)                                                                          | When False Negatives (missing depressed students) are more dangerous, e.g., in health applications. |
| **F1 Score**                | Moderate                                  | Moderate                                  | \( 2 \times \frac{Precision \times Recall}{Precision + Recall} \)                              | When there’s a need for a balance between False Positives and False Negatives (e.g., in classification tasks where both errors matter). |
| **Specificity**             | Yes (minimizes False Positives)            | No                                         | \( \frac{TN}{TN + FP} \)                                                                          | When False Positives lead to unnecessary interventions or stigmatization (e.g., in cases where a false alarm is costly). |
| **True Positive Rate (TPR)**| No                                         | Yes (same as Recall)                       | \( \frac{TP}{TP + FN} \)                                                                          | When it's crucial to identify all actual positives (e.g., identifying all depressed students). |
| **Area Under ROC Curve (AUC-ROC)** | Yes (reduces False Positives)            | Yes (reduces False Negatives)              | AUC-ROC involves plotting the True Positive Rate (TPR) vs. False Positive Rate (FPR) at various thresholds. TPR is \( \frac{TP}{TP + FN} \), FPR is \( \frac{FP}{FP + TN} \). | When a balance between both errors is needed and you're interested in assessing overall model performance across thresholds. |
| **Precision-Recall AUC**    | Yes (helps focus on reducing False Positives) | Yes (helps focus on reducing False Negatives) | Similar to AUC-ROC but specifically focuses on Precision and Recall.                           | When the data is imbalanced, and the goal is to minimize both types of errors while handling imbalanced classes effectively. |


Now lets compare accuracies of the models

In [99]:
print('Test Accuracy using Logistic Regression:',test_accuracy)
print('Difference:',round(abs(test_accuracy-train_accuracy),2))
print('Test Accuracy of DTC using GS:',test_accuracy_DTC_gs)
print('Difference:',round(abs(test_accuracy_DTC_gs-train_accuracy_DTC_gs),2))
print('Test Accuracy of LR using GS:',test_accuracy_LR_gs)
print('Difference:',round(abs(test_accuracy_LR_gs-train_accuracy_LR_gs),2))
print('Test Accuracy of RFC using GS:',test_accuracy_RFC_gs)
print('Difference:',round(abs(test_accuracy_RFC_gs-train_accuracy_RFC_gs),2))
print('Test accuracy using SVC:',svc_test_accuracy)
print('Difference:',np.round(abs(svc_test_accuracy-svc_train_accuracy),2))
print('Test accuracy for XGB:',XGB_test_accuracy)
print('Difference:', np.round(abs(XGB_test_accuracy-XGB_train_accuracy),2))

Test Accuracy using Logistic Regression: 81.87
Difference: 0.79
Test Accuracy of DTC using GS: 80.56
Difference: 3.85
Test Accuracy of LR using GS: 81.87
Difference: 0.81
Test Accuracy of RFC using GS: 81.75
Difference: 2.38
Test accuracy using SVC: 81.78
Difference: 0.88
Test accuracy for XGB: 81.65
Difference: 2.08


In [100]:
from sklearn.metrics import recall_score
print('Recall for LR:',np.round(recall_score(y_test,y_pred)*100,2))
print('Recall for DTC using GS:',np.round(recall_score(y_test,y_pred_DTC_test_gs)*100,2))
print('Recall for LR using GS:',np.round(recall_score(y_test,y_pred_LR_test_gs)*100,2))
print('Recall for SVC:',np.round(recall_score(y_test,y_pred_svc_test)*100,2))
print('Recall for XGB:',np.round(recall_score(y_test,y_pred_XGB_test)*100,2))

Recall for LR: 86.69
Recall for DTC using GS: 84.31
Recall for LR using GS: 86.63
Recall for SVC: 88.59
Recall for XGB: 86.52


Our SVC model has second highest accuracy and third highest in terms of overfitting. But as it is a depression related data and type-1 error is important, meaning we have to minimize the false negatives as much as we can. So we calculated the recall score and SVC came highest in it. So we will be selecting SVC as our model.